# 04 - Model Benchmark Sweep - Averaging over entire day

In [3]:
!pip install pymongo catboost

  Using cached pymongo-4.17.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached catboost-1.2.10-cp312-cp312-manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 18.3 MB/s eta 0:00:00


In [8]:
import os
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display
from pymongo import MongoClient
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import tensorflow as tf

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

load_dotenv(str(ROOT / ".env"), override=False)

def load_colab_secrets(keys: list[str]) -> None:
    try:
        from google.colab import userdata
    except Exception:
        return
    for key in keys:
        if os.getenv(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value

load_colab_secrets(["MONGO_URI", "MONGO_DB_NAME"])

sns.set_theme(style="whitegrid")

@dataclass
class ModelConfig:
    name: str
    type: str
    params: dict


MODEL_CONFIGS = [
    ModelConfig(name="lightgbm", type="lgbm", params= {"n_estimators": 200,"learning_rate": 0.03,"num_leaves": 15,"min_child_samples": 50,"subsample": 0.6,"colsample_bytree": 0.6,"reg_alpha": 1.0,"reg_lambda": 5.0,"n_jobs": -1,"verbose": -1,}),
    ModelConfig(name="xgboost", type="xgb", params={"n_estimators": 300, "learning_rate": 0.05, "max_depth": 6}),
    ModelConfig(name="catboost", type="cat", params={"iterations": 400, "learning_rate": 0.05, "depth": 6}),
    ModelConfig(name="random_forest", type="rf", params={"n_estimators": 400, "max_depth": 20, "n_jobs": -1}),
    ModelConfig(name="extra_trees", type="extra", params={"n_estimators": 400, "max_depth": 20, "n_jobs": -1}),
    ModelConfig(name="gradient_boosting", type="gbr", params={"n_estimators": 300, "learning_rate": 0.05}),
    ModelConfig(name="ridge_regression", type="ridge", params={"alpha": 1.0}),
    ModelConfig(name="linear_regression", type="linreg", params={}),
    ModelConfig(name="gru", type="gru", params={"units": 64, "epochs": 15, "batch_size": 64}),
    ModelConfig(name="lstm", type="lstm", params={"units": 64, "epochs": 15, "batch_size": 64}),
]

TOP_FEATURES = [
    "wind_speed_10m_rolling_std_168h",
    "hour_sin",
    "is_day",
    "hour_of_day",
    "solar_radiation_category",
    "european_aqi_rolling_std_168h",
    "day_of_week_cos",
    "hour_cos",
    "european_aqi_rolling_std_24h",
    "european_aqi_rolling_min_168h",
    "pm10_rolling_mean_24h",
    "hour_traffic_weight",
    "days_since_last_rain",
    "relative_humidity_2m_rolling_min_24h",
    "pm10_rolling_mean_48h",
    "european_aqi_rolling_std_48h",
    "day_of_week",
    "pm10_rolling_mean_12h",
    "pm10_rolling_std_168h",
    "european_aqi_rolling_mean_3h",
    "day_of_week_sin",
    "european_aqi_rolling_min_3h",
    "pm2_5_rolling_mean_6h",
    "precipitation_cumulative_72h",
    "nitrogen_dioxide_lag_6h",
    "pm10_rolling_std_24h",
    "pm2_5_rolling_min_3h",
    "is_evening_rush",
    "pm2_5_rolling_min_6h",
    "pm2_5_rolling_mean_12h",
    "oxidant_index",
    "european_aqi_rolling_std_12h",
    "pressure_change_6h",
    "nitrogen_dioxide_lag_1h",
    "pm2_5_rolling_mean_3h",
    "pm10_rolling_min_3h",
    "relative_humidity_2m_rolling_std_24h",
    "weekend_traffic_factor",
    "european_aqi_lag_1h",
    "wind_speed_10m_rolling_min_48h",
]

ARTIFACTS_DIR = Path("/content/debug_exports")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

horizon = 3
DAILY_HOURS = 24
best_window_days = 180

SEED = 42
SPLIT_RATIO = 0.9
TARGET_MODEL_COUNT = 10
MAX_DL_MODELS = 2
DL_TYPES = {"gru", "lstm"}
TREE_MODEL_NAMES = {"lightgbm", "xgboost", "catboost", "random_forest", "extra_trees", "gradient_boosting"}

CPU_OVERRIDES = {
    "xgboost": {"tree_method": "hist", "n_jobs": 1, "random_state": SEED},
    "catboost": {"task_type": "CPU", "thread_count": 1, "random_seed": SEED},
    "extra_trees": {"n_jobs": 1, "random_state": SEED},
    "random_forest": {"n_jobs": 1, "random_state": SEED},
    "lightgbm": {"device_type": "cpu", "n_jobs": 1, "random_state": SEED},
    "gradient_boosting": {"random_state": SEED},
}

BENCHMARK_MODEL_NAMES = [config.name for config in MODEL_CONFIGS]

def apply_cpu_overrides(config: ModelConfig) -> ModelConfig:
    overrides = CPU_OVERRIDES.get(config.name, {})
    if not overrides:
        return config
    params = {**config.params, **overrides}
    return ModelConfig(name=config.name, type=config.type, params=params)


def get_mongo_client() -> MongoClient:
    uri = os.getenv("MONGO_URI")
    if not uri:
        raise ValueError("MONGO_URI is required")
    return MongoClient(uri)


def get_database():
    client = get_mongo_client()
    db_name = os.getenv("MONGO_DB_NAME", "aqi_predictor")
    return client.get_database(db_name)


def evaluate_forecast(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"rmse": rmse, "mae": mae, "r2": r2}


def per_horizon_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    metrics = {}
    labels = ["day1", "day2", "day3"]

    for idx, label in enumerate(labels):
        if idx < y_true.shape[1]:
            yt = y_true[:, idx]
            yp = y_pred[:, idx]
            metrics[f"rmse_{label}"] = float(np.sqrt(mean_squared_error(yt, yp)))
            metrics[f"mae_{label}"]  = float(mean_absolute_error(yt, yp))
            metrics[f"r2_{label}"]   = float(r2_score(yt, yp))

    return metrics

def _build_dl_model(model_type: str, input_shape: Tuple[int, int], output_steps: int, units: int):
    model = tf.keras.Sequential()
    if model_type == "gru":
        model.add(tf.keras.layers.GRU(units, input_shape=input_shape, return_sequences=False))
    else:
        model.add(tf.keras.layers.LSTM(units, input_shape=input_shape, return_sequences=False))
    model.add(tf.keras.layers.Dense(128, activation="relu"))
    model.add(tf.keras.layers.Dense(output_steps))
    model.compile(optimizer="adam", loss="mse")
    return model


def _prepare_sequence_data(
    features: pd.DataFrame,
    target: pd.Series,
    lookback: int,
):
    X, y, start_indices = [], [], []

    values = features.values
    target_values = target.values

    required_future = 72

    for idx in range(lookback, len(features) - required_future):

        day1 = target_values[idx : idx + 24].mean()
        day2 = target_values[idx + 24 : idx + 48].mean()
        day3 = target_values[idx + 48 : idx + 72].mean()

        X.append(values[idx - lookback : idx])
        y.append([day1, day2, day3])
        start_indices.append(idx)

    return (
        np.array(X),
        np.array(y),
        np.array(start_indices),
    )


def _fit_single_output(config: ModelConfig, x_train: np.ndarray, y_train: np.ndarray):
    """Fit one regressor for a single day target."""
    if config.type == "lgbm":
        base = lgb.LGBMRegressor(**config.params)
    elif config.type == "xgb":
        base = xgb.XGBRegressor(**config.params)
    elif config.type == "cat":
        base = cb.CatBoostRegressor(**config.params, verbose=False)
    elif config.type == "rf":
        base = RandomForestRegressor(**config.params)
    elif config.type == "extra":
        base = ExtraTreesRegressor(**config.params)
    elif config.type == "ridge":
        base = Ridge(**config.params)
    elif config.type == "linreg":
        base = LinearRegression(**config.params)
    else:
        base = GradientBoostingRegressor(**config.params)
    base.fit(x_train, y_train)
    return base


def train_model(
    config: ModelConfig,
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_val: pd.DataFrame,
    y_val: pd.DataFrame,
    horizon: int = 3,
    feature_frame: pd.DataFrame = None,
    target: pd.Series = None,
    split_index: int = None,
):
    # ─────────────────────────────────────────────
    # Deep Learning models (GRU / LSTM)
    # ─────────────────────────────────────────────
    if config.type in DL_TYPES:
        if feature_frame is None or target is None or split_index is None:
            raise ValueError("feature_frame, target, and split_index are required for DL models")

        lookback = 24
        X_seq, y_seq, start_indices = _prepare_sequence_data(
            feature_frame, target, lookback,
        )

        train_mask = start_indices + 72 <= split_index
        val_mask   = start_indices >= split_index

        if not train_mask.any() or not val_mask.any():
            raise ValueError("Not enough sequences for the requested train/validation split")

        X_train_seq, X_val_seq = X_seq[train_mask], X_seq[val_mask]
        y_train_seq, y_val_seq = y_seq[train_mask], y_seq[val_mask]

        model = _build_dl_model(
            config.type,
            input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]),
            output_steps=3,
            units=config.params.get("units", 64),
        )
        model.fit(
            X_train_seq, y_train_seq,
            epochs=config.params.get("epochs", 10),
            batch_size=config.params.get("batch_size", 32),
            verbose=0,
        )

        train_preds = model.predict(X_train_seq)
        val_preds   = model.predict(X_val_seq)
        return model, train_preds, val_preds, y_train_seq, y_val_seq

    # ─────────────────────────────────────────────
    # Classical ML — chained: day1 → day2 → day3
    # ─────────────────────────────────────────────
    x_tr = x_train.values
    x_vl = x_val.values
    y_tr = y_train.values   # shape (n_train, 3)
    y_vl = y_val.values     # shape (n_val,   3)

    # ── Day 1 ──────────────────────────────────
    m1 = _fit_single_output(config, x_tr, y_tr[:, 0])
    pred_tr_d1 = m1.predict(x_tr).reshape(-1, 1)
    pred_vl_d1 = m1.predict(x_vl).reshape(-1, 1)

    # ── Day 2  (features + predicted day1) ─────
    x_tr_d2 = np.hstack([x_tr, pred_tr_d1])
    x_vl_d2 = np.hstack([x_vl, pred_vl_d1])
    m2 = _fit_single_output(config, x_tr_d2, y_tr[:, 1])
    pred_tr_d2 = m2.predict(x_tr_d2).reshape(-1, 1)
    pred_vl_d2 = m2.predict(x_vl_d2).reshape(-1, 1)

    # ── Day 3  (features + predicted day1 + predicted day2) ──
    x_tr_d3 = np.hstack([x_tr, pred_tr_d1, pred_tr_d2])
    x_vl_d3 = np.hstack([x_vl, pred_vl_d1, pred_vl_d2])
    m3 = _fit_single_output(config, x_tr_d3, y_tr[:, 2])
    pred_tr_d3 = m3.predict(x_tr_d3).reshape(-1, 1)
    pred_vl_d3 = m3.predict(x_vl_d3).reshape(-1, 1)

    # ── Stack outputs → (n, 3) ─────────────────
    train_preds = np.hstack([pred_tr_d1, pred_tr_d2, pred_tr_d3])
    val_preds   = np.hstack([pred_vl_d1, pred_vl_d2, pred_vl_d3])

    # Bundle models so caller can save them if needed
    model = (m1, m2, m3)

    return model, train_preds, val_preds, y_tr, y_vl


def build_window_frame(frame: pd.DataFrame, days: int) -> pd.DataFrame:
    latest_timestamp = frame["timestamp"].max()
    window_start = latest_timestamp - pd.Timedelta(days=days)
    return frame.loc[frame["timestamp"] >= window_start].sort_values("timestamp").reset_index(drop=True)


def prepare_training_matrices(
    feature_frame: pd.DataFrame,
    target: pd.Series,
    split_index: int,
):
    required_future = 72

    if feature_frame.empty or len(feature_frame) <= required_future:
        return None, None, None, None, None

    max_start = len(target) - required_future

    if max_start <= 0 or split_index <= required_future or split_index >= max_start:
        return None, None, None, None, None

    x = feature_frame.iloc[:max_start].copy()

    y = []

    for i in range(max_start):

        day1 = target.iloc[i : i + 24].mean()
        day2 = target.iloc[i + 24 : i + 48].mean()
        day3 = target.iloc[i + 48 : i + 72].mean()

        y.append([day1, day2, day3])

    y = pd.DataFrame(y, index=x.index)

    start_indices = np.arange(max_start)

    train_mask = start_indices + required_future <= split_index
    val_mask = start_indices >= split_index

    if not train_mask.any() or not val_mask.any():
        return None, None, None, None, None

    return (
        x.iloc[train_mask],
        y.iloc[train_mask],
        x.iloc[val_mask],
        y.iloc[val_mask],
        list(x.columns),
    )

db = get_database()
collection = db["aqi_features_rawalpindi"]
data = pd.DataFrame(list(collection.find()))

if data.empty:
    raise ValueError("aqi_features_rawalpindi is empty")

if "_id" in data.columns:
    data = data.drop(columns=["_id"])

data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
data = data.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

feature_cols_all = [column for column in data.columns if column != "timestamp"]
feature_cols = [feature for feature in TOP_FEATURES if feature in data.columns] or feature_cols_all

window_frame = build_window_frame(data, best_window_days)
split_index = int(len(window_frame) * SPLIT_RATIO)

if split_index <= 72:
    raise ValueError("Not enough rows to create a leakage-safe split")

target = window_frame["european_aqi"].astype(float)
feature_frame = window_frame[feature_cols].apply(pd.to_numeric, errors="coerce").ffill().fillna(0.0)
feature_frame = feature_frame.loc[:, feature_frame.nunique(dropna=True) > 1].copy()

x_train, y_train, x_val, y_val, used_features = prepare_training_matrices(feature_frame, target, split_index)

if x_train is None or y_train is None or x_val is None or y_val is None:
    raise ValueError("Unable to prepare benchmark matrices for the selected window and split")

In [9]:
results = []
benchmark_model_names = BENCHMARK_MODEL_NAMES
model_lookup = {config.name: config for config in MODEL_CONFIGS}

missing_models = [name for name in benchmark_model_names if name not in model_lookup]
if missing_models:
    print(f"Skipping missing model configs: {missing_models}")

benchmark_models = [
    apply_cpu_overrides(model_lookup[name])
    for name in benchmark_model_names
    if name in model_lookup
]

if not benchmark_models:
    raise ValueError("No benchmark models available for the configured shortlist")

rows_after_prep = len(x_train) + len(x_val)

for config in benchmark_models:
    print(f"\nRunning benchmark for {config.name}...")

    try:
        model, train_preds, val_preds, y_train_true, y_val_true = train_model(
            config,
            x_train,
            y_train,
            x_val,
            y_val,
            horizon=horizon,
            feature_frame=feature_frame,
            target=target,
            split_index=split_index,
        )

        # ─────────────────────────────────────────────
        # Metrics: TRAIN
        # ─────────────────────────────────────────────
        train_metrics = evaluate_forecast(y_train_true, train_preds)
        train_metrics.update(per_horizon_metrics(y_train_true, train_preds))

        # ─────────────────────────────────────────────
        # Metrics: VALIDATION
        # ─────────────────────────────────────────────
        val_metrics = evaluate_forecast(y_val_true, val_preds)
        val_metrics.update(per_horizon_metrics(y_val_true, val_preds))

        # ─────────────────────────────────────────────
        # Logging
        # ─────────────────────────────────────────────
        print(
            f"{config.name} | "
            f"Train RMSE: {train_metrics['rmse']:.4f} | "
            f"Val RMSE: {val_metrics['rmse']:.4f} | "
            f"Val R2: {val_metrics['r2']:.4f} | "
            f"Day1: {val_metrics.get('r2_day1', float('nan')):.4f} | "
            f"Day2: {val_metrics.get('r2_day2', float('nan')):.4f} | "
            f"Day3: {val_metrics.get('r2_day3', float('nan')):.4f}"
        )

        results.append(
            {
                "model": config.name,
                "type": config.type,
                "window_days": best_window_days,
                "rows_after_prep": rows_after_prep,
                "features_used": len(used_features),

                # TRAIN
                "train_rmse": train_metrics["rmse"],
                "train_mae": train_metrics["mae"],
                "train_r2": train_metrics["r2"],
                "train_r2_day1": train_metrics.get("r2_day1"),
                "train_r2_day2": train_metrics.get("r2_day2"),
                "train_r2_day3": train_metrics.get("r2_day3"),

                # VALIDATION
                "val_rmse": val_metrics["rmse"],
                "val_mae": val_metrics["mae"],
                "val_r2": val_metrics["r2"],
                "val_r2_day1": val_metrics.get("r2_day1"),
                "val_r2_day2": val_metrics.get("r2_day2"),
                "val_r2_day3": val_metrics.get("r2_day3"),

                # Overfitting signal
                "overfit_gap_rmse": val_metrics["rmse"] - train_metrics["rmse"],
            }
        )

    except Exception as exc:
        print(f"Model {config.name} failed: {exc}")

        results.append(
            {
                "model": config.name,
                "type": config.type,
                "window_days": best_window_days,
                "rows_after_prep": rows_after_prep,
                "features_used": len(used_features),

                "train_rmse": float("nan"),
                "train_mae": float("nan"),
                "train_r2": float("nan"),
                "train_r2_day1": float("nan"),
                "train_r2_day2": float("nan"),
                "train_r2_day3": float("nan"),

                "val_rmse": float("nan"),
                "val_mae": float("nan"),
                "val_r2": float("nan"),
                "val_r2_day1": float("nan"),
                "val_r2_day2": float("nan"),
                "val_r2_day3": float("nan"),

                "overfit_gap_rmse": float("nan"),
                "error": str(exc),
            }
        )

results_df = pd.DataFrame(results)
results_path = ARTIFACTS_DIR / "model_benchmark_metrics.csv"
results_df.to_csv(results_path, index=False)

valid_results = results_df.dropna(subset=["val_rmse"]).copy()

if valid_results.empty:
    raise ValueError("All benchmark runs failed")

valid_results = valid_results.sort_values(
    ["val_rmse", "model"],
    na_position="last"
).reset_index(drop=True)

best_row = valid_results.iloc[0]
best_model_name = str(best_row["model"])

best_tree_results = valid_results[valid_results["model"].isin(TREE_MODEL_NAMES)].copy()
if not best_tree_results.empty:
    best_tree_results = best_tree_results.sort_values(
        ["val_rmse", "model"],
        na_position="last"
    ).reset_index(drop=True)
    best_tree_model_name = str(best_tree_results.iloc[0]["model"])
else:
    best_tree_model_name = best_model_name

with open(ARTIFACTS_DIR / "best_model_name.txt", "w", encoding="utf-8") as f:
    f.write(best_model_name)

with open(ARTIFACTS_DIR / "best_tree_model_name.txt", "w", encoding="utf-8") as f:
    f.write(best_tree_model_name)

with open(ARTIFACTS_DIR / "best_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_model_name": best_model_name,
            "best_model_type": str(best_row["type"]),
            "best_window_days": best_window_days,
            "val_rmse": float(best_row["val_rmse"]),
            "val_mae": float(best_row["val_mae"]),
            "val_r2": float(best_row["val_r2"]),
            "val_r2_day1": float(best_row.get("val_r2_day1", float("nan"))),
            "val_r2_day2": float(best_row.get("val_r2_day2", float("nan"))),
            "val_r2_day3": float(best_row.get("val_r2_day3", float("nan"))),
            "rows_after_prep": int(best_row["rows_after_prep"]),
            "features_used": int(best_row["features_used"]),
        },
        f,
        indent=2,
    )

display(results_df)

print(f"\nSaved {results_path}")
print(f"Best model (by validation RMSE): {best_model_name}")
print(f"Best tree model: {best_tree_model_name}")


Running benchmark for lightgbm...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

lightgbm | Train RMSE: 2.1492 | Val RMSE: 11.7883 | Val R2: 0.4804 | Day1: 0.7790 | Day2: 0.3572 | Day3: 0.3049

Running benchmark for xgboost...
xgboost | Train RMSE: 0.3916 | Val RMSE: 14.1277 | Val R2: 0.2551 | Day1: 0.6982 | Day2: 0.0603 | Day3: 0.0068

Running benchmark for catboost...
catboost | Train RMSE: 1.1436 | Val RMSE: 12.1310 | Val R2: 0.4467 | Day1: 0.7378 | Day2: 0.3330 | Day3: 0.2693

Running benchmark for random_forest...
random_forest | Train RMSE: 0.3565 | Val RMSE: 14.2311 | Val R2: 0.2505 | Day1: 0.7309 | Day2: 0.1098 | Day3: -0.0893

Running benchmark for extra_trees...
extra_trees | Train RMSE: 0.0050 | Val RMSE: 11.8008 | Val R2: 0.4784 | Day1: 0.7812 | Day2: 0.3130 | Day3: 0.3410

Running benchmark for gradient_boosting...
gradient_boosting | Train RMSE: 2.3555 | Val RMSE: 12.8325 | Val R2: 0.3885 | Day1: 0.7804 | Day2: 0.2094 | Day3: 0.1758

Running benchmark for ridge_regression...
ridge_regression | Train RMSE: 7.3374 | Val RMSE: 13.6093 | Val R2: 0.3152 | 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
gru | Train RMSE: 6.8395 | Val RMSE: 14.1312 | Val R2: 0.2323 | Day1: 0.4824 | Day2: 0.1929 | Day3: 0.0217

Running benchmark for lstm...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
lstm | Train RMSE: 6.4819 | Val RMSE: 12.7981 | Val R2: 0.3678 | Day1: 0.5444 | Day2: 0.3816 | Day3: 0.1774


,model,type,window_days,rows_after_prep,features_used,train_rmse,train_mae,train_r2,train_r2_day1,train_r2_day2,train_r2_day3,val_rmse,val_mae,val_r2,val_r2_day1,val_r2_day2,val_r2_day3,overfit_gap_rmse
0,lightgbm,lgbm,180,4178,40,2.149222,1.650262,0.980088,0.984062,0.977069,0.979133,11.788315,8.549001,0.480372,0.779023,0.357236,0.304858,9.639092
1,xgboost,xgb,180,4178,40,0.391642,0.283603,0.999340,0.999287,0.999364,0.999368,14.127686,10.257614,0.255091,0.698176,0.060308,0.006789,13.736044
2,catboost,cat,180,4178,40,1.143636,0.881359,0.994366,0.994749,0.993881,0.994469,12.131047,8.657592,0.446681,0.737791,0.332954,0.269299,10.987411
3,random_forest,rf,180,4178,40,0.356474,0.183667,0.999455,0.999148,0.999588,0.999629,14.231093,10.408184,0.250454,0.730885,0.109809,-0.089332,13.874619
4,extra_trees,extra,180,4178,40,0.004962,0.002765,1.000000,1.000000,1.000000,1.000000,11.800828,8.338164,0.478367,0.781167,0.312958,0.340978,11.795866
5,gradient_boosting,gbr,180,4178,40,2.355497,1.793736,0.976071,0.982814,0.971038,0.974359,12.832519,9.284839,0.388549,0.780401,0.209420,0.175827,10.477021
6,ridge_regression,ridge,180,4178,40,7.337369,5.357799,0.767204,0.926077,0.721953,0.653580,13.609275,9.555732,0.315219,0.759822,0.184965,0.000869,6.271906
7,linear_regression,linreg,180,4178,40,7.337349,5.357806,0.767205,0.926080,0.721953,0.653581,13.609316,9.556438,0.315244,0.760066,0.184961,0.000706,6.271968
8,gru,gru,180,4178,40,6.839507,5.192667,0.798189,0.867870,0.810133,0.716563,14.131193,10.112331,0.232311,0.482361,0.192905,0.021666,7.291686
9,lstm,lstm,180,4178,40,6.481913,4.824482,0.818729,0.892425,0.819045,0.744717,12.798097,9.205228,0.367788,0.544365,0.381575,0.177425,6.316184



Saved /content/debug_exports/model_benchmark_metrics.csv
Best model (by validation RMSE): lightgbm
Best tree model: lightgbm
